In [ ]:
import faulthandler, glob, json, os, platform, subprocess, sys, threading, time
T0 = time.time()
faulthandler.enable(all_threads=True)   # a segfault / abort prints every thread's traceback (commit v1 died without one: an OOM kill prints nothing)


def monitor(every=120):                 # process RSS, MemAvailable, GPU memory (nvidia-smi: no CUDA call from this thread)
    while True:
        time.sleep(every)
        try:
            rss = int(next(l for l in open("/proc/self/status") if l.startswith("VmRSS")).split()[1]) / 2**20
            av = int(next(l for l in open("/proc/meminfo") if l.startswith("MemAvailable")).split()[1]) / 2**20
            gpu = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True,
                                 timeout=20).stdout.split()
            print(f"[mon {time.time() - T0:.0f}s] rss {rss:.1f} GB, MemAvailable {av:.1f} GB, gpu used {gpu} MiB", flush=True)
        except Exception as e:
            print(f"[mon] {type(e).__name__}: {e}", flush=True)


threading.Thread(target=monitor, daemon=True).start()
import numpy as np, pandas as pd, torch
SLUGS = ['knee-oct21-fa-base', 'knee-oct21-fa-d1', 'knee-oct21-fa-d5', 'knee-oct21-fa-cfg-inv1-a']   # the variant's datasets: frozen WEIGHTS datasets (union) + its small CONFIG dataset (last: code + config.json)
INPUT = os.environ.get("KNEE_INPUT", "/kaggle/input")  # local dry run: a folder laid out like /kaggle/input
COMP = next(p for p in [INPUT + "/competitions/rsna-knee-abnormality-detection", INPUT + "/rsna-knee-abnormality-detection"]
            if os.path.exists(p + "/test.csv"))
# oct21_final_alldata (FINAL all-data package, variant inv1_a; OAI: no): the weights live in frozen datasets (uploaded once, shared by
# every variant), code + config in the variant's config dataset. Every dataset's file set and every file's md5 are checked against the
# locally audited build (make_kernel.py: audit_inv1_a.json PASS for this config.json) before anything runs; a file name may live in
# ONE dataset only.
import hashlib, shutil
MD5_MODE = {'knee-oct21-fa-base': 'full', 'knee-oct21-fa-d1': 'full', 'knee-oct21-fa-d5': 'full', 'knee-oct21-fa-cfg-inv1-a': 'full'}   # per dataset: 'full' = md5 of every file; 'ends' = big files: size + md5 of the first / last 4 MB (--md5 / --md5-light)
EXPECT = {'knee-oct21-fa-base': {'basin_nfnet_all_A1.json': '65363c57b1becbaa9566de4f525269e7', 'basin_nfnet_all_A1.pt': '47137b4672f2a425929ba52ce275185f', 'basin_nfnet_all_A2.json': '65a6ceca4d11ac0d000838d09eeecb76', 'basin_nfnet_all_A2.pt': '820f546caaaeb622ebb7a21466ef1768', 'basin_nfnet_all_A3.json': '0cabeca7b1fbcf88a7ca7611ba34ccbc', 'basin_nfnet_all_A3.pt': '84647915c7be750363b664c97e769ebf', 'basin_nfnet_all_A4.json': 'fc8ab2dcb77ab98c80abbf2f2e4d3f54', 'basin_nfnet_all_A4.pt': 'b2d522c0f99e6589e343088a9925a32b', 'basin_nfnet_all_A5.json': '0498d94f2ba8ef5eeefb60033ca55e6b', 'basin_nfnet_all_A5.pt': 'bac7fa1a3ed137500cd81e8d5592a712', 'basin_nfnet_sd_all_s101.json': '65363c57b1becbaa9566de4f525269e7', 'basin_nfnet_sd_all_s101.pt': '6c8a5c25c288040964c52b3f9b44d28f', 'basin_nfnet_sd_all_s102.json': '65363c57b1becbaa9566de4f525269e7', 'basin_nfnet_sd_all_s102.pt': 'ffccf08d3752e83452bdcde25105c202', 'basin_rad_all_A1.json': '80f1b32ad463138ab44155a433218fc3', 'basin_rad_all_A1.pt': 'b612758863cdcb5d5b3049232cad4f63', 'basin_rad_all_A2.json': '44afe3f978bf0caa2497c0dcc5348dbf', 'basin_rad_all_A2.pt': 'c8d94195056974da3abc38e9ed4ac352', 'basin_rad_all_A3.json': '1ae153b65cbbfccadedb5489a83ace9a', 'basin_rad_all_A3.pt': '41130c8d57df78c8821228273f6d3382', 'basin_rad_all_A4.json': 'e049c2b02127ef8d33693710ff18c016', 'basin_rad_all_A4.pt': '0f92f062755bea465b40f4bf73c4493f', 'basin_rad_all_A5.json': '3fb6034242627ea08353c9ffc9d3f026', 'basin_rad_all_A5.pt': '158062db34a5422abdfac10a5e0853d2', 'basinsyn_nfnet_all_A1.json': '1198036e4990dbcc1121b1a48552fc62', 'basinsyn_nfnet_all_A1.pt': 'e717eb7f2b9fa68754b2435d5ab98bc9', 'basinsyn_nfnet_all_A2.json': 'c88180da5ff904233d45d399ff064b15', 'basinsyn_nfnet_all_A2.pt': '5faefe1646d561390cc50368bbada042', 'basinsyn_nfnet_all_A3.json': '75dd5a02be507d7d12879224f619f6c5', 'basinsyn_nfnet_all_A3.pt': '2d937dcbb3eb7e91b235f904bf986f2c', 'fc_b3c_rep_all_s101.pt': 'f686dd16e3556f2b3fb39284fb4f5964', 'fc_b3c_rep_all_s102.pt': 'dc03c637870c47285360ff884a3cc861', 'fc_b3l_rep_all_s101.pt': '5d0e56b0e2a923de630ed036c870395b', 'fc_b3l_rep_all_s102.pt': '5838141605f747bb7a0e97025e2ad5c4', 'fc_b3n_rep_all_s101.pt': 'b8627e69cec6bda01a2330676f1b168a', 'fc_b3n_rep_all_s102.pt': '18fd2f188743412797e864973f399d20', 'fc_bfs_rep_all_s101.pt': '87c06c15a7bd74a9c25bc62406423f35', 'fc_bfs_rep_all_s102.pt': '2a1ca9e15691978700e33d5c63331bf7', 'fc_loc.pt': '13717d66243e99d1387f1177da9f0cb4', 'flipdet.pt': '7adacf7a8258fb382bfe274f06e92b9e', 'm_prob05_all_A1.pt': '5e68c45917cb6fcc89c381987e9f2c0c', 'm_prob05_all_A2.pt': '6badffa3b97609bab255254361c697ea', 'm_prob05_all_A3.pt': '9b931863375379c1de741d0b8fb6efdc', 'main_sd_all_s101.pt': 'db1811706034930690a37125461f6f6a', 'oa_flipdet.pt': '7adacf7a8258fb382bfe274f06e92b9e', 'oa_loc.pt': '26b7659ce4c4b5376a44f3adb64ad14a', 'oa_oa_crop_v2cor_all_A1.pt': 'c328f462ce56621d6bf35b1710937bb1', 'oa_oa_crop_v2cor_all_A2.pt': '558ad4e60ddeb7bca4889e55897dc75a', 'oa_oa_crop_v2cor_all_A3.pt': '8dbc43adba945395a6f2d9dc46336b70', 'oa_oa_crop_v2cor_all_A4.pt': '7162c0368a322f47e2fefaf8484a9cbe', 'oa_oa_crop_v2cor_all_A5.pt': '60cb05b4a88305033dbbba7e6480fdfb', 'oa_oa_crop_v2cor_nfnet_all_A1.pt': 'fb7e20abf86ccf4ae6a57f993db06eaf', 'oa_oa_crop_v2cor_nfnet_all_A2.pt': '5ab6f63b6f7cc60063ec792ec3c9729e', 'oa_oa_crop_v2cor_nfnet_all_A3.pt': '6618e953c60c6a2eedf36b953cdb52d8', 'oa_oa_oasep_all_s101.pt': '9a7d09f207fdbab29a1410322de84a6b', 'oa_oa_oasep_all_s102.pt': '1035ebe80fa4dece1596da6431e8cfeb', 'oa_oa_oasep_nf_all_s101.pt': 'da73162350a32ae5cbffc70c960fc510', 'oa_oa_oasep_nf_all_s102.pt': '37642f804bc7ee308e15c80f8e0a9f3d', 'oa_oa_silsep_all_s101.pt': '830101bda698ec7ec1e1231580c71e69', 'oa_oa_silsep_all_s102.pt': '3c72f667136d902c1128106e9bb33acb', 'sub_b0_all_A1.pt': '42cf65632eeea34f75acfe8bcc10186e', 'sub_b0_all_A2.pt': 'a68c702608cc89ae05ea00852048e3fe', 'sub_b0_all_A3.pt': '432b93a4b8ae688dcc6710d500bdfc80', 'sub_b0_all_A4.pt': 'd824afb4e6a93720373d66e9ee70d2df', 'sub_b0_all_A5.pt': '365aa3659f10714c97167fa868ec4cab', 'syn_b3g_all_s101.pt': '62e1545fa8868f14766bffdcec9b2711', 'syn_b3g_all_s102.pt': '95b2fdf592ab61c8872b2971582af29d', 'syn_cs_fc125_all_s101.pt': '8141f681df3bcf5f5ff650e8b8793ee3', 'syn_cs_fc125_all_s103.pt': 'cd1bd8f877b7fe4bf65494e031afdfb8', 'syn_cs_fs_all_s101.pt': '202b61791cc276a476b05f9b145f3e4b', 'syn_ef3_all_s101.pt': '517a9c2cf07cd877caa656bf6ed748d0', 'syn_ef3_all_s102.pt': '76cc88dd835ac34f1c916af7990720cc', 'syn_ef3fc_all_s101.pt': 'a3aa1352b0ea4165bc5ddf04a8baf263', 'syn_ef3fc_all_s102.pt': '7812318cdd75d55b43689670e013d23a', 'syn_ef3nf_all_s101.pt': '408ce3e7694e5cfffebac8549ae5f414', 'syn_ef3nf_all_s102.pt': '503febb3bf76152640637f41ac7deb8e', 'syn_ef3nffc_all_s101.pt': 'cb081b7c2a49083938e02df12701ec53', 'syn_ef3nffc_all_s102.pt': '681fc8395b899060a0222d4cf9270678', 'syn_s_fc125_nf_all_A1.pt': '81909d60c55ce66b5620693814c09bbe', 'syn_s_fc125_nf_all_A2.pt': 'e626555b2975fa32b42b376f131ee733', 'syn_s_fc125_nf_all_A3.pt': '678373d38e14018a9a2f32f1c39cff01', 'syn_s_fs_nf_all_A1.pt': '1e23eaf2cec2b93806a57a297a8b1f22', 'syn_s_fs_nf_all_A2.pt': 'e41557514baff04dab98481e2f2f1b47', 'syn_s_fs_nf_all_A3.pt': '6f9f15cead6b1515eb049ffca7c1cdd3', 'syn_w5b_mcl_all_s101.pt': '47fbc38285437400ebff65df2ee776c8', 'syn_w5b_mcl_all_s102.pt': '918c31fc457969b16c9b6e21fbf31a2f', 'syn_z1_acl_all_s101.pt': '8f9e588d0d269afcea4e4271489950bb', 'syn_z1_acl_all_s102.pt': '521bcf9f2dd3f1451a8b728b6260d5ea'}, 'knee-oct21-fa-d1': {'basin_nfnet_sd_all_s103.json': '65363c57b1becbaa9566de4f525269e7', 'basin_nfnet_sd_all_s103.pt': '5d22f953b432ced1fe9d8cfaff9253ed', 'fc_b3c_rep_all_s103.pt': '44bb83c1f7c0a18c5c37e3c19fab3cd2', 'fc_b3l_rep_all_s103.pt': 'dd731a931e4ee89a3fef9adb41721035', 'fc_b3n_rep_all_s103.pt': 'adc15ff718845415be747109214ba949', 'fc_bfs_rep_all_s103.pt': '670b23ca021071e89c2843e275ce7be0', 'fc_bfs_rep_all_s104.pt': 'f11619085495c898a2d74a774e69b4c4', 'fc_bfs_rep_all_s105.pt': '414e0b6fe6da13ae91766bb4cd1456cf', 'fc_bfs_rep_all_s106.pt': 'c42a0088daccc92e2ae0741c5e6f4033', 'fc_bfs_rep_all_s107.pt': '42d36333f19b1e8cbec2d5edf5b542af', 'main_sd_all_s102.pt': 'b95983c3f690332b964895ed251a1c2d', 'main_sd_all_s103.pt': 'fa86adb92720d09387a6fa0433b02478', 'main_sd_all_s104.pt': '933cf1fbb08473f58768ffa8c4e5c09f', 'main_sd_all_s105.pt': '95df27b73492657f9c2b56126a7a9480', 'oa_oa_crop_v2cor_all_A104.pt': '62976051ea75d53c3e51d6117f16edad', 'oa_oa_crop_v2cor_nfnet_all_A104.pt': 'fc7f5db5bd1dfcdb339ab99921d50b36', 'oa_oa_crop_v2cor_nfnet_all_A105.pt': 'f49832ba975d7c832bc64c4aa8c4ee51', 'oa_oa_oasep_all_s103.pt': '9096f61b7e80c4e34c7a3e4f762a9088', 'oa_oa_oasep_nf_all_s103.pt': 'be0bc8fe7459b133a66fe4effa29ae2f', 'oa_oa_silsep_all_s103.pt': '928b94dd2a6b08b4d933311f2378db6d', 'sub_b0_all_A104.pt': '66bb6503bb92e59a41596d896c9c4974', 'syn_b3g_all_s103.pt': 'e42b321cb744897b8f240da5b57cf64b', 'syn_b3g_all_s104.pt': '95dd52ebcad19b072ef6d50cf4803e1f', 'syn_cs_fc125_all_s102.pt': '2ac8e326986aa585916b7daf7a6861d0', 'syn_cs_fs_all_s102.pt': '6a6dd664b460cdf078c5c0002eafb333', 'syn_cs_fs_all_s103.pt': '9799521b83b1ebca5005c27de3c8d8d7', 'syn_ef3_all_s103.pt': '518d8e7153201c270e7ba1bdfb9dad06', 'syn_ef3fc_all_s103.pt': 'fa531a7067aa2106a78e8b859c39340f', 'syn_ef3nf_all_s103.pt': '546c0d005662d23dcbcdc35d730f7c64', 'syn_ef3nffc_all_s103.pt': 'd4d85758085c210ab826eca95f9f8abd', 'syn_w5b_mcl_all_s103.pt': 'f6884b556bcbbf6d3330cac07f3e9b7b', 'syn_w5b_mcl_all_s104.pt': 'd752a1316119e0d42a276d3868a3d54a', 'syn_w5b_mcl_all_s105.pt': '72d27ced358d87d27b3c620072601a20', 'syn_w5b_mcl_all_s106.pt': '6b519a24a8f38c32bc6a8d4b66bfebff', 'syn_w5b_mcl_all_s107.pt': '55ef9c25d70fa2fd46e641f983555d8e', 'syn_z1_acl_all_s103.pt': 'd708e72b0b191d41d7601c89270cc490', 'syn_z1_acl_all_s104.pt': 'f7fec29069ca37ddfa3928e898d0ad05', 'syn_z1_acl_all_s105.pt': 'b967a27111e0c05bed66e9bffc5205ee', 'syn_z1_acl_all_s106.pt': 'c786b4b91b2547dac68d9e0fe19294f4', 'syn_z1_acl_all_s107.pt': '7dd6b266181bc3decb046a06e5c1ccf2'}, 'knee-oct21-fa-d5': {'syn_w5b_mcl_all_s108.pt': '4914d9f273d1e82b6d4a4c7bbf4573ad', 'syn_w5b_mcl_all_s109.pt': '24dc923802ae274194b671db9b382033'}, 'knee-oct21-fa-cfg-inv1-a': {'basin_flipdet.py': '2acbb496ffaeaedb3119965e117e8806', 'basin_model.py': '6726ce60d0452bf512c150414d16ba0a', 'basin_predict.py': 'e761e2f35490d6af3ef8a45bcd0578f8', 'check_studies.csv': '7fc8717ceb4eca2f1d1db77f30b566e5', 'common_geometry.py': '0cffaaf23b7ce6473ba7afa9cbf2397b', 'common_meta.py': 'cc8312f20cb4db4611947b70da3056ca', 'common_paths.py': '343949cb095a2ae28f327593120e7607', 'config.json': '3262165f39ba86213ef7d74d68af28d1', 'ensemble.py': '7467a24e4aaada2da2c057a8d7f5c2c0', 'fc_stage.py': 'e09c1b48ae965315bc79bcf56afa888d', 'infer_prep.py': 'ad30d9405ef57aab4605862be8cbab00', 'knee_infer.py': 'c993446cc7ca91e300c223cdb9ee1cd1', 'main_sd_config.json': 'f3c03d64c4623db904af2bc88dcaf980', 'model.py': '5b2ac31bcd864b479a7cb98d9d6a14df', 'oa_stage.py': 'd4b13314e21910d996b69e93d5cf8fde', 'prep.py': 'eebaa8e20edeae0865f951e9e270705f', 'slots.py': '79be9a2523bbda36fcfd602075b1adcf', 'syn_members_stack.json': '700f1c5033abad74336649ab29d7f523', 'syn_stage.py': '825fc6246d86ee6e756edd1dabd132e0', 'win_stage.py': 'd6c00dc5a9e3288a7aa4d3ef4a7bbd07'}}


def _fp(p, mode):
    """'full' = md5 of the file; 'ends' = a file > 64 MB: its size + md5 of its first and last 4 MB (smaller files: the full md5)."""
    n = os.path.getsize(p)
    h = hashlib.md5()
    with open(p, "rb") as f:
        if mode == "ends" and n > 64 << 20:
            h.update(f.read(4 << 20))
            f.seek(n - (4 << 20))
            h.update(f.read(4 << 20))
            return f"{n}:{h.hexdigest()}"
        for b in iter(lambda: f.read(1 << 24), b""):
            h.update(b)
    return h.hexdigest()



def _mount(slug):
    hits = sorted({p for pat in ("/*", "/*/*", "/*/*/*") for p in glob.glob(INPUT + pat) if os.path.isdir(p) and os.path.basename(p) == slug})
    assert len(hits) == 1, f"dataset {slug}: mount dirs {hits}"
    return hits[0]


MOUNT = {s: _mount(s) for s in SLUGS}
_names = [f for s in SLUGS for f in EXPECT[s]]
assert list(EXPECT) == SLUGS and len(_names) == len(set(_names)), "a file name in two datasets"
_t5 = time.time()
for _s in SLUGS:
    _have = set(os.listdir(MOUNT[_s])) - {"__pycache__", "dataset-metadata.json"}
    assert _have == set(EXPECT[_s]), f"{_s}: files != the audited build: extra {sorted(_have - set(EXPECT[_s]))[:5]}, missing {sorted(set(EXPECT[_s]) - _have)[:5]}"
LOCAL = os.environ.get("KNEE_LOCAL", "/tmp/pkgw")   # --local-copy (Submission Timing 21:1x): every package file copied to local disk with its
shutil.rmtree(LOCAL, ignore_errors=True)                # md5 computed on the fly (ONE read of gcsfuse, full md5 of everything); WORK links there,
os.makedirs(LOCAL)                                      # so every (re)load reads local disk (~300+ MB/s) instead of gcsfuse (~5 MB/s cold)
from concurrent.futures import ThreadPoolExecutor


def _cp(job):
    src, dst = job
    h = hashlib.md5()
    with open(src, "rb", buffering=0) as fi, open(dst, "wb") as fo:
        for b in iter(lambda: fi.read(1 << 24), b""):
            h.update(b)
            fo.write(b)
    return os.path.basename(dst), h.hexdigest()


with ThreadPoolExecutor(16) as _ex:
    _got = dict(_ex.map(_cp, [(os.path.join(MOUNT[s], f), os.path.join(LOCAL, f)) for s in SLUGS for f in EXPECT[s]]))
_bad = [f for s in SLUGS for f in EXPECT[s] if _got[f] != EXPECT[s][f]]
assert not _bad, f"package md5 != the audited build (local copy): {_bad[:10]}"
MOUNT_SRC = {s: LOCAL for s in SLUGS}
MD5_S = time.time() - _t5
CFGD = MOUNT[SLUGS[-1]]
_cfg = json.load(open(CFGD + "/config.json"))
_ad = _cfg["all_data"]
assert _ad["variant"] == 'inv1_a' and _ad["datasets"] == SLUGS and set(_ad["runs"]) == set(_ad["sources"]) and set(_ad["runs"]) <= set(_names), \
    "all_data variant / datasets / runs"
_seeds = {r: sorted({str(x["seed"]) for x in _ad["runs"].values() if x["recipe"] == r}) for r in sorted({x["recipe"] for x in _ad["runs"].values()})}
print(f"FINAL package inv1_a: {len(SLUGS)} datasets {SLUGS}, {len(_names)} files md5 == the audited build (MD5 {MD5_MODE} wall {MD5_S:.0f} s, "
      f"{sum(os.path.getsize(os.path.join(MOUNT[s], f)) for s in SLUGS for f in EXPECT[s]) / 1e9:.1f} GB); form {_ad['form']}; seeds per recipe "
      f"{_seeds}", flush=True)
# WORK = the union of the datasets (code copied so knee_infer's sys.path stays in WORK, the rest symlinked) + the MCL SD-2 main pass dir
# (config mcl_sd main_dir) rebuilt from main_sd_config.json; config and code are used unchanged.
WORK = "/kaggle/working/pkg_final" if os.path.isdir("/kaggle/working") else os.path.join(os.environ.get("TMPDIR", "/tmp"), "pkg_final")
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(os.path.join(WORK, _cfg["mcl_sd"]["main_dir"]))
for _s in SLUGS:
    for fn in EXPECT[_s]:
        (shutil.copy if fn.endswith(".py") else os.symlink)(os.path.join(MOUNT_SRC[_s], fn), os.path.join(WORK, fn))
_sdc = json.load(open(CFGD + "/main_sd_config.json"))
json.dump(_sdc, open(os.path.join(WORK, _cfg["mcl_sd"]["main_dir"], "config.json"), "w"))
for fn in _sdc["weights"]:
    os.symlink(os.path.realpath(os.path.join(WORK, fn)), os.path.join(WORK, _cfg["mcl_sd"]["main_dir"], fn))
PKG = WORK
_sm = json.load(open(PKG + "/" + _cfg["syn"]["members"]))
_em = _cfg.get("eff_main") or {}
print("RUNTIME models per stage:", {"main": len(_cfg["weights"]), "main_sd": len(_sdc["weights"]),
      "basin": len(set(sum(list(_cfg["basin"].values()) + list(_cfg["basin_syn"].values()) + [_cfg["mcl_sd"]["nfnet"]] + list(_em.get("basin", {}).values()), []))),
      "win_main": len(_cfg["win"]["main"]) + len(_em.get("main", [])), "oa": sum(len(m["files"]) for m in _cfg["oa"]["members"].values()),
      "fc": sum(len(m["files"]) for m in _cfg["fc"]["members"].values()),
      "syn": sum(len(m["files"]) for m in _sm.values()) + len((_cfg.get("cont_sxm") or {}).get("files", []))}, "| fc members", list(_cfg["fc"]["members"]),
      "| fc_shipped_off", bool(_cfg.get("fc_shipped_off")), "| frac_allmem", bool(_cfg.get("frac_allmem")), "| eff_main", bool(_em),
      "| cont_sxm", bool(_cfg.get("cont_sxm")), flush=True)
sys.path.insert(0, PKG)
import cv2, pydicom, timm
import ensemble
ki = ensemble.ki
for _m in (ensemble, ensemble.syn_stage, ensemble.oa_stage, ensemble.fc_stage, ki):     # the final package's code is what runs (fail fast otherwise)
    assert os.path.dirname(os.path.abspath(_m.__file__)) == WORK, _m.__file__
_es = open(ensemble.__file__).read()
assert all(k in _es for k in ("mcl_member2", "mcl_sd", "acl_member2", "loa_extra_align", "eff_pipelines", "cont_fs", "cont_allmem", "frac_allmem",
                              "frac_fs", "eff_main", "cont_sxm", "fc_shipped_off", "def basin_args")) \
    and "keep_bags" in open(ensemble.syn_stage.__file__).read() and "def member_sets" in open(ensemble.fc_stage.__file__).read()
print("final modules loaded from", WORK, flush=True)
NCPU = len(os.sched_getaffinity(0))
print(f"python {platform.python_version()} torch {torch.__version__} timm {timm.__version__} pydicom {pydicom.__version__} cv2 {cv2.__version__} "
      f"numpy {np.__version__} pandas {pd.__version__} | cpus {NCPU} gpus {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]} "
      f"| package {PKG} | weights {json.load(open(PKG + '/config.json'))['weights']} basin {json.load(open(PKG + '/config.json'))['basin']}", flush=True)

DEC = {'MCL': {'context': 'Contusion', 'b': 0.2, 'c0': -2.0172928736727602, 'c': 0.31754119357063526}, 'Lateral Meniscus': {'context': 'ACL', 'b': 0.1, 'c0': -1.371208412532161, 'c': 0.322331809283129}}                                          # post-hoc de-coupling z' = z - b (c0 + c z_ctx): MCL Q6 RE-FITTED on the new Contusion (refit.json), LM Q22


def decouple(df):
    out = df.copy()
    for f, d in DEC.items():
        p, q = np.clip(df[f].to_numpy(np.float64), 1e-7, 1 - 1e-7), np.clip(df[d["context"]].to_numpy(np.float64), 1e-7, 1 - 1e-7)
        z, zc = np.log(p / (1 - p)), np.log(q / (1 - q))
        out[f] = 1 / (1 + np.exp(-(z - d["b"] * (d["c0"] + d["c"] * zc))))
    return out


test = pd.read_csv(COMP + "/test.csv")
t = time.time()
# ===== FINAL fold-avg package + FAST two-GPU runner, STRICT (Master 10-20, submissions/oct20_final_v2/make_final.py; OAI: no).
# USER: "we absolutely do not need any fallbacks. we need robust inference pipeline that works or fails loudly".
import gc
open("/kaggle/working/fast.py", "w").write('"""Two-GPU runner, port to the merged stacked fold package submissions/oct20_stack_all (from submissions/oct19_f5_win_fast, which was\nbit-identical to the 0.961 package on Kaggle). The workers take their stage arguments from the package\'s ensemble.py hooks\n(basin_args / oa_cfg / fc_cfg / syn_members: the same functions ensemble.predict calls), so every switch (frac_mfs / pf_q20 / bakers_allpw)\ngives identical stage calls. Dev runs against oct19_stack_fold use a hook shim (env FAST_HOOKS). Serial test mode (env FAST_SERIAL=1):\nthe two workers run one after another on GPU 0 (local parity on a 1-GPU machine; same outputs as two GPUs of one type).\n\nOriginal header (0.961 runner): Predictions = the package\'s own.\n\nThe package runs its stages one after another, and four of them use one T4 only (Basin, OA experts, bone experts, synovitis crops: ~90 % of\nits 262 min). Here two worker processes (CUDA_VISIBLE_DEVICES 0 / 1) each run those four stages on half of the studies, at the same time.\nThe halves follow Basin\'s study order (the series csv order) and split at an even index, so Basin\'s 2-study batches are the package\'s.\nknee_infer (main) and the window stage already use both GPUs: they stay in the parent. The parent then calls the package\'s unchanged\nensemble.predict with the four stage functions returning the workers\' merged outputs, so the combination code is the package\'s own.\nOne difference by construction: Basin\'s fallback for a study without any readable series (cohort mean logit) is the mean over that\nworker\'s half instead of over all studies (never triggered on the 275 check studies of 10-13).\n\nProfiling (prof=True): every stage call\'s start / end (parent and workers), and the time inside the crop stages\' building blocks\n(knee_infer.study_series in the DataLoader workers, localise, study_bags, study_tensors, correct_sides, expert / fsmain logits);\nread with read_prof(). Timing wrappers only: no change to any call\'s arguments or results.\n\nRAM (10-19, after the ON bench\'s host OOM): every worker stage starts and ends with release() (gc, CUDA cache, pinned-host cache, glibc\narenas; RSS logged in the worker timings), the crop loop releases the pinned-host cache every RELEASE_EVERY studies, pad_loc 32 is the\ndefault, and every predict() call spawns fresh worker processes that exit at its end (all their memory returns to the OS).\n\n  worker: python fast.py worker PKG ROOT JOB.pkl OUT.pkl"""\nimport contextlib\nimport json\nimport os\nimport pickle\nimport resource\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\n\nimport pandas as pd\n\nSTAGE = ["parent"]          # current stage label; DataLoader workers inherit it at fork\nPROF_DIR = os.environ.get("KNEE_PROF_DIR", "/kaggle/working/prof")\nSTAGE_LOG = []              # (process, stage, t_start, t_end)\nRELEASE_EVERY = 16          # crop loop: return the pinned-host cache / glibc arenas every N studies (10-19 RAM fix; no effect on outputs)\nPARENT_RELEASE_EVERY = 16   # parent: the same for knee_infer\'s main passes (10-19 Kaggle parent_mem: 25 GB of cached pinned study buffers on 325 studies)\n\n\ndef rss_gb():\n    return int(next(l for l in open("/proc/self/status") if l.startswith("VmRSS")).split()[1]) / 2**20\n\n\n@contextlib.contextmanager\ndef parent_release(ki, every=PARENT_RELEASE_EVERY):\n    """Parent-side pinned-cache release (10-19; Kaggle parent_mem: knee_infer\'s cached pinned study buffers reached 25 GB in one main pass of\n    325 heavy studies, all returned by the package\'s own _host_emptyCache AFTER the pass). Every `every` knee_infer.load_study calls (they run\n    in knee_infer\'s loader threads) the cache\'s FREE pinned blocks are returned; blocks in use or with pending copy events stay (the host\n    allocator is mutex-protected and frees only unused, event-complete blocks). Memory only: outputs unchanged."""\n    import torch\n    orig, n, lock = ki.load_study, [0], threading.Lock()\n\n    def wrapped(*a, **k):\n        r = orig(*a, **k)\n        with lock:\n            n[0] += 1\n            go = every and n[0] % every == 0\n        if go and torch.cuda.is_available():\n            torch._C._host_emptyCache()\n        return r\n    ki.load_study = wrapped\n    try:\n        yield n\n    finally:\n        ki.load_study = orig\n\n\ndef release(full=True):\n    """Return freed memory (10-19 RAM fix after the ON bench\'s host OOM; no effect on any output): Python garbage (full), the CUDA caching\n    allocator (full), the pinned-host cache (grows with every new pinned buffer size: knee_infer pins each study\'s DICOM bytes) and glibc\'s\n    malloc arenas."""\n    import ctypes\n    import gc\n    import torch\n    if full:\n        gc.collect()\n    if torch.cuda.is_available():\n        if full:\n            torch.cuda.empty_cache()\n        torch._C._host_emptyCache()\n    ctypes.CDLL("libc.so.6").malloc_trim(0)\nCROP_FUNCS = ["localise", "study_bags", "study_tensors", "correct_sides", "expert_logits", "fsmain_logits"]\n\n\ndef _mods(pkg):\n    sys.path.insert(0, pkg)\n    import basin_predict as bp\n    import fc_stage\n    import knee_infer as ki\n    import oa_stage\n    import syn_stage\n    import win_stage\n    return bp, ki, oa_stage, fc_stage, syn_stage, win_stage\n\n\ndef _hooks(pkg):\n    """The package\'s stage-argument hooks (ensemble.py). FAST_HOOKS=path/to/shim.py only for dev against a package without them."""\n    sys.path.insert(0, pkg)\n    import importlib.util\n    shim = os.environ.get("FAST_HOOKS")\n    if shim:\n        spec = importlib.util.spec_from_file_location("fast_hooks", shim)\n        mod = importlib.util.module_from_spec(spec)\n        spec.loader.exec_module(mod)\n        return mod\n    import ensemble\n    assert all(hasattr(ensemble, h) for h in ("basin_args", "oa_cfg", "fc_cfg", "syn_members")), "ensemble.py lacks the stage hooks"\n    return ensemble\n\n\ndef _append(name, sec):\n    os.makedirs(PROF_DIR, exist_ok=True)\n    with open(f"{PROF_DIR}/prof_{os.getpid()}.tsv", "a") as fh:\n        fh.write(f"{STAGE[0]}\\t{name}\\t{sec:.6f}\\n")\n\n\ndef _timed(fn, name):\n    def w(*a, **k):\n        t = time.time()\n        try:\n            return fn(*a, **k)\n        finally:\n            _append(name, time.time() - t)\n    w.__wrapped__ = fn\n    return w\n\n\ndef _stage(fn, label, proc):\n    def w(*a, **k):\n        prev, STAGE[0] = STAGE[0], label\n        t = time.time()\n        try:\n            return fn(*a, **k)\n        finally:\n            STAGE_LOG.append((proc, label, t, time.time()))\n            STAGE[0] = prev\n    w.__wrapped__ = fn\n    return w\n\n\ndef _timed_sync(fn, name):\n    """Timing wrapper that synchronises the GPU before and after (profiling runs only: adds two syncs per call)."""\n    import torch\n\n    def w(*a, **k):\n        torch.cuda.synchronize()\n        t = time.time()\n        try:\n            return fn(*a, **k)\n        finally:\n            torch.cuda.synchronize()\n            _append(name, time.time() - t)\n    w.__wrapped__ = fn\n    return w\n\n\ndef install_prof(pkg, proc, sync=False):\n    """Timing wrappers on the package modules (idempotent). sync: also Basin\'s gpu_views / export_windows / model forwards with GPU\n    syncs (their GPU time; changes scheduling, not results)."""\n    bp, ki, oa_stage, fc_stage, syn_stage, win_stage = _mods(pkg)\n    if sync and getattr(bp.export_windows, "__wrapped__", None) is None:\n        bp.export_windows = _timed_sync(bp.export_windows, "basin.export_windows")\n        bp.gpu_views = _timed_sync(bp.gpu_views, "basin.gpu_views")\n        orig_load = bp.load_model\n\n        def load_model(ckpt):\n            m, cfg = orig_load(ckpt)\n            m.forward = _timed_sync(m.forward, "basin.model_forward")\n            return m, cfg\n        bp.load_model = load_model\n    if getattr(ki.study_series, "__wrapped__", None) is None:\n        ki.study_series = _timed(ki.study_series, "study_series")\n        for mod in (oa_stage, fc_stage, syn_stage):\n            for f in CROP_FUNCS:\n                if hasattr(mod, f):\n                    setattr(mod, f, _timed(getattr(mod, f), f))\n        for mod, f, label in ((bp, "predict", "basin"), (ki, "predict", "main"), (win_stage, "main_windows", "win"),\n                              (oa_stage, "predict", "oa"), (fc_stage, "predict_pkg", "fc"), (syn_stage, "predict", "syn")):\n            setattr(mod, f, _stage(getattr(mod, f), label, proc))\n\n\ndef install_pad_loc(pkg, bucket):\n    """The crop stages\' localiser (a per-slice MobileNetV3 + MLP head, BatchNorm in eval mode) on series batches padded to a multiple of\n    `bucket` slices (zero rows, outputs dropped): a few fixed batch shapes instead of one per series length (Kaggle\'s cuDNN builds kernels\n    for every new shape; bench v1: the OA stage\'s localise took 78 s for 101 calls, the same calls 2.6 s in later stages). Real rows\'\n    outputs can differ only by cuDNN\'s per-shape fp32 algorithm choice. NOT exact: a separate bench variant."""\n    import cv2\n    import numpy as np\n    import torch\n    bp, ki, oa_stage, fc_stage, syn_stage, _ = _mods(pkg)\n    for mod in (oa_stage, fc_stage, syn_stage):\n        @torch.no_grad()\n        def localise(loc, vol, plane, pos, dev, mod=mod):\n            n = len(vol)\n            small = np.stack([cv2.resize(a, (mod.LOC_PX, mod.LOC_PX), interpolation=cv2.INTER_AREA) for a in vol])\n            w = np.stack([small[np.clip(np.arange(n) + d, 0, n - 1)] for d in (-1, 0, 1)], 1)\n            aux = mod.aux_features(plane, pos, n)\n            m = -(-n // bucket) * bucket\n            if m > n:\n                w = np.concatenate([w, np.zeros((m - n,) + w.shape[1:], w.dtype)])\n                aux = np.concatenate([aux, np.zeros((m - n,) + aux.shape[1:], aux.dtype)])\n            p, b = loc(torch.from_numpy(w).to(dev), torch.from_numpy(aux).to(dev))\n            return torch.sigmoid(p[:n]).float().cpu().numpy(), b[:n].float().cpu().numpy()\n        mod.localise = localise\n\n\ndef read_prof():\n    """-> DataFrame stage x function -> total seconds, calls (all processes\' files)."""\n    rows = []\n    for p in Path(PROF_DIR).glob("prof_*.tsv"):\n        d = pd.read_csv(p, sep="\\t", names=["stage", "func", "sec"])\n        d["pid"] = p.stem[5:]\n        rows.append(d)\n    if not rows:\n        return pd.DataFrame()\n    d = pd.concat(rows)\n    return d.groupby(["stage", "func"]).sec.agg(["sum", "count"]).round(2)\n\n\ndef worker(pkg, root, job, out):\n    """The package\'s four single-GPU stage calls (arguments as ensemble.predict builds them) on this worker\'s studies."""\n    t_enter = time.time()\n    J = pickle.load(open(job, "rb"))\n    if J.get("pad_loc"):\n        install_pad_loc(pkg, J["pad_loc"])\n    if J["prof"]:\n        install_prof(pkg, J["name"], sync=J.get("prof_sync", False))\n    bp, ki, oa_stage, fc_stage, syn_stage, win_stage = _mods(pkg)\n    import torch\n    T = {"t_enter": t_enter, "import_s": time.time() - t_enter}       # spawn -> enter = t_enter - FAST["t0"] (per-call overhead)\n    FINDINGS = ki.FINDINGS\n    cfg = json.load(open(Path(pkg) / "config.json"))\n    studies, workers = J["studies"], J["workers"]\n    series = J["series"]\n    series = series[series.StudyInstanceUID.isin(studies)].assign(Fat_Suppression=lambda d: d.Fat_Suppression.fillna(0).astype(int))\n    fs_of = dict(zip(series.SeriesInstanceUID, series.Fat_Suppression))\n    R = {}\n\n    H = _hooks(pkg)\n\n    def basin():\n        ckpts, det, win_kw = H.basin_args(cfg, pkg)\n        if not len(series):\n            return {}, dict(bp.WIN), dict(bp.STATS)\n        import dicom_prefetch as DP   # --readahead (Master 10-22, safe3w9ra): Basin\'s upcoming studies read into the page cache by a\n        pf = DP.start(root, list(dict.fromkeys(series.StudyInstanceUID)), threads=64, ahead=24, tag=f"basin_{J[\'name\']}")   # separate\n        unhook = DP.hook_getitem(bp.DicomStudies, pf)   # process (exact: reads only); hooked BEFORE Basin\'s DataLoader forks\n        try:\n            z = bp.predict(series, root, ckpts, workers=workers, detector=det, win=win_kw)\n        finally:\n            unhook()\n            st = DP.stop(pf)   # performance-only: a dead prefetcher = Basin reads as before (same outputs), logged in T\n            T.update({f"pf_{k}": float(v) for k, v in st.items() if isinstance(v, (int, float))})\n            print(f"READAHEAD {J[\'name\']}: {st}", flush=True)\n        return z, dict(bp.WIN), dict(bp.STATS)\n\n    def oa():\n        return oa_stage.predict(root, studies, fs_of, pkg, H.oa_cfg(cfg), workers=workers, threads=None), dict(oa_stage.STATS)\n\n    def fc():\n        return fc_stage.predict_pkg(root, studies, fs_of, pkg, H.fc_cfg(cfg), workers=workers), dict(fc_stage.STATS)\n\n    def syn():\n        s = cfg["syn"]\n        members = H.syn_members(cfg, pkg)\n        zm = pd.DataFrame(syn_stage.predict(root, studies, fs_of, members, str(Path(pkg) / s["loc"]), str(Path(pkg) / s["detector"]),\n                                            workers=workers, fuse=s.get("fuse")))\n        return zm, dict(syn_stage.STATS)\n\n    run = {"basin": basin, "oa": oa, "fc": fc, "syn": syn}\n    have = {"basin": True, "oa": bool(cfg.get("oa")) and H.oa_cfg(cfg)["mode"] != "main", "fc": bool(cfg.get("fc")), "syn": bool(cfg.get("syn"))}\n    order = J.get("order", ["basin", "oa", "fc", "syn"])\n    if J.get("merge_crops"):              # one study loop for the three crop stages (crops), at the position of the first crop stage\n        assert all(have[s] for s in ("oa", "fc", "syn")), "merge_crops needs the oa, fc and syn stages"\n        first = min(order.index(s) for s in ("oa", "fc", "syn"))\n        order = [s for s in order if s not in ("oa", "fc", "syn")]\n        order.insert(first, "crops")\n        have["crops"] = True\n        run["crops"] = lambda: crop_all(pkg, root, studies, fs_of, cfg, workers)\n    for st in order:\n        if have[st]:\n            release()                         # the previous stage\'s models / buffers are unreferenced locals by now: give their memory back\n            T[f"{st}_rss_in_gb"] = rss_gb()\n            t = time.time()\n            r = run[st]()\n            if st == "crops":\n                R.update(r)\n                T["crops_load_s"] = r["oa"][1]["load_s"]\n            else:\n                R[st] = r\n            T[st] = time.time() - t\n            T[f"{st}_rss_out_gb"] = rss_gb()\n            release()\n            T[f"{st}_rss_released_gb"] = rss_gb()\n    T["rss_peak_gb"] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20\n    T["rss_peak_loader_gb"] = resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 2**20\n    T["gpu_peak_gb"] = torch.cuda.max_memory_reserved() / 2**30\n    pickle.dump({"R": R, "T": T, "stage_log": STAGE_LOG}, open(out, "wb"))\n\n\ndef crop_all(pkg, root, studies, fs_of, cfg, workers):\n    """oa_stage.predict + fc_stage.predict_pkg + syn_stage.predict in ONE study loop: each study\'s DICOM is decoded once (one\n    DataLoader), the shared mirror detector (one weight file, identical correct_sides in the three modules) runs once, then each stage\'s\n    own localiser and experts run as in its predict(), with that stage\'s failure handling. The bone stage\'s main-type members still see\n    the uncorrected series. -> {"oa": (out, STATS), "fc": (out, STATS), "syn": (DataFrame, STATS)} in the stages\' return formats."""\n    import collections\n\n    import numpy as np\n    import torch\n    bp, ki, oa_stage, fc_stage, syn_stage, win_stage = _mods(pkg)\n    from basin_flipdet import FlipNet\n    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    t0 = time.time()\n    H = _hooks(pkg)\n    ocfg, fcfg, scfg = H.oa_cfg(cfg), H.fc_cfg(cfg), cfg["syn"]\n    dets = {str(Path(pkg) / ocfg["detector"]), str(Path(pkg) / fcfg["detector"]), str(Path(pkg) / scfg["detector"])}\n    blobs = {open(p, "rb").read() for p in dets}\n    assert len(blobs) == 1, "merge_crops needs one detector for the three crop stages"\n    assert oa_stage.correct_sides.__code__.co_code == fc_stage.correct_sides.__code__.co_code == syn_stage.correct_sides.__code__.co_code \\\n        or all(hasattr(f, "__wrapped__") for f in (oa_stage.correct_sides, fc_stage.correct_sides, syn_stage.correct_sides))\n    det = FlipNet(pretrained=False)\n    det.load_state_dict(torch.load(sorted(dets)[0], map_location="cpu"))\n    det = det.to(dev).eval()\n    # OA (as oa_stage.predict)\n    assert all(m["kind"] == "expert" for m in ocfg["members"].values()), "merge_crops: OA main-type members not supported"\n    cor_v2 = ocfg.get("loc_version", "v1") == "v2cor"\n    structs = oa_stage.STRUCTS_V2 if cor_v2 else oa_stage.STRUCTS\n    oloc = oa_stage.Loc(structs=structs)\n    oloc.load_state_dict(torch.load(f"{pkg}/{ocfg[\'loc\']}", map_location="cpu"))\n    oloc = oloc.to(dev).to(memory_format=torch.channels_last).eval()\n    oexp = oa_stage.load_experts(pkg, ocfg, dev)\n    xsets = {k: frozenset(v) for k, v in (ocfg.get("extra_sets") or {}).items()}     # oct20: oa_stage.predict\'s extra fusions\n    orows = {k: {} for k in list(oexp) + (["fused"] if ocfg.get("fuse") else []) + (["fused_loa_extra"] if ocfg.get("loa_extra") else [])\n             + [f"fused_{k}" for k in xsets]}\n    okw = {"extra_sets": xsets} if xsets else {}\n    # bone (as fc_stage.predict_pkg / predict)\n    fmem = {n: {"kind": e["kind"], "cfg": e["cfg"], "files": [str(Path(pkg) / f) for f in e["files"]]} for n, e in fcfg["members"].items()}\n    floc = fc_stage.Loc()\n    floc.load_state_dict(torch.load(str(Path(pkg) / fcfg["loc"]), map_location="cpu"))\n    floc = floc.to(dev).to(memory_format=torch.channels_last).eval()\n    fexp, fsm = fc_stage.load_experts(fmem, dev), fc_stage.load_fsmain(fmem, dev)\n    frows = {k: {} for k in list(fexp) + list(fsm) + ([fc_stage.fused_key(f) for f in sorted({f for e in fmem.values() if e["kind"] == "bone" and not e["cfg"].get("nofuse", False) for f in fc_stage.member_sets(e["cfg"])})] if fcfg.get("fuse") else [])}   # oct20_probe_cont_allmem: every fusion group\n    # synovitis (as syn_stage.predict)\n    smem = H.syn_members(cfg, pkg)\n    sloc = syn_stage.Loc()\n    sloc.load_state_dict(torch.load(str(Path(pkg) / scfg["loc"]), map_location="cpu"))\n    sloc = sloc.to(dev).to(memory_format=torch.channels_last).eval()\n    sexp = syn_stage.load_experts(smem, dev)\n    srows = collections.defaultdict(dict)\n    S = {k: {"failed": [], "flipped": 0} for k in ("oa", "fc", "syn")}\n    load_s = time.time() - t0                  # the three stages\' models + localisers + detector loaded (per-call overhead)\n    dl = torch.utils.data.DataLoader(oa_stage.Studies(root, studies, fs_of), batch_size=None, num_workers=workers, collate_fn=lambda x: x,\n                                     prefetch_factor=2 if workers else None)\n    for i_study, (sid, series, err) in enumerate(dl):\n        if RELEASE_EVERY and i_study and i_study % RELEASE_EVERY == 0:\n            release(full=False)\n        if err or not series:\n            for k in S:\n                S[k]["failed"].append(sid)\n            print(f"crops: study {sid} failed ({err or \'no readable series\'})", flush=True)\n            continue\n        try:\n            for k, v in fc_stage.fsmain_logits(fsm, series, dev).items():   # main-type bone members: before the side correction (as trained)\n                frows[k][sid] = v\n        except Exception as e:  # noqa: BLE001\n            S["fc"]["failed"].append(sid)\n            print(f"bone_stage: study {sid} failed ({type(e).__name__}: {e})", flush=True)\n            fc_failed = True\n        else:\n            fc_failed = False\n        try:\n            n_flip = oa_stage.correct_sides(series, det, dev)\n        except Exception as e:  # noqa: BLE001\n            for k in S:\n                if not (k == "fc" and fc_failed):\n                    S[k]["failed"].append(sid)\n            print(f"crops: study {sid} failed at the side correction ({type(e).__name__}: {e})", flush=True)\n            continue\n        for k in S:\n            S[k]["flipped"] += n_flip\n        try:\n            bags = oa_stage.study_bags(series, oloc, dev, structs, cor_v2)\n            for k, v in oa_stage.expert_logits(oexp, series, bags, dev, ocfg.get("fuse"), set(ocfg.get("loa_extra", [])),\n                                               set(ocfg.get("loa_set", [])), **okw).items():   # = the stack\'s oa_stage.predict call\n                orows[k][sid] = v\n        except Exception as e:  # noqa: BLE001\n            S["oa"]["failed"].append(sid)\n            print(f"oa_stage: study {sid} failed ({type(e).__name__}: {e}); OA keeps the pipeline blend", flush=True)\n        if not fc_failed:\n            try:\n                locs = [fc_stage.localise(floc, s["vol"], s["plane"], s["pos"], dev) if len(s["vol"]) >= 3 else None for s in series]\n                for k, v in fc_stage.expert_logits(fexp, series, locs, dev, fcfg.get("fuse")).items():\n                    frows[k][sid] = v\n            except Exception as e:  # noqa: BLE001\n                S["fc"]["failed"].append(sid)\n                print(f"bone_stage: study {sid} failed ({type(e).__name__}: {e})", flush=True)\n        try:\n            locs = [syn_stage.localise(sloc, s["vol"], s["plane"], s["pos"], dev) if len(s["vol"]) >= 3 else None for s in series]\n            for k, v in syn_stage.expert_logits(sexp, series, locs, dev, scfg.get("fuse")).items():\n                srows[k][sid] = v\n        except Exception as e:  # noqa: BLE001\n            S["syn"]["failed"].append(sid)\n            print(f"syn_stage: study {sid} failed ({type(e).__name__}: {e})", flush=True)\n    wall = time.time() - t0\n    for k in S:\n        S[k]["wall_s"], S[k]["load_s"] = wall, load_s\n    oout = {k: pd.DataFrame.from_dict(v, orient="index", columns=oa_stage.OUTS).astype(np.float64) for k, v in orows.items()}\n    fout = {k: pd.DataFrame.from_dict(v, orient="index", columns=["Fracture", "Contusion"]) for k, v in frows.items()}\n    sout = pd.DataFrame({k: pd.Series(v, dtype=np.float64) for k, v in srows.items()})\n    return {"oa": (oout, S["oa"]), "fc": (fout, S["fc"]), "syn": (sout, S["syn"])}\n\n\ndef split(studies, series):\n    """Two study lists: Basin\'s order (series csv order of the studies), cut at an even index; studies without series rows go to the\n    first list (Basin never sees them; the crop stages log them as failed, as in the package)."""\n    order = list(dict.fromkeys(series.StudyInstanceUID[series.StudyInstanceUID.isin(studies)]))\n    k = (len(order) + 1) // 2\n    k += k % 2\n    missing = [s for s in studies if s not in set(order)]\n    return [order[:k] + missing, order[k:]]\n\n\ndef _merge(parts):\n    """Worker outputs -> one stage output per stage, in the package\'s return formats."""\n    out = {}\n    zs = [p["R"]["basin"] for p in parts]\n    keys = list(dict.fromkeys(k for z, _, _ in zs for k in z))\n    out["basin"] = ({k: pd.concat([z[k] for z, _, _ in zs if k in z]) for k in keys},\n                    {s: v for _, w, _ in zs for s, v in w.items()},\n                    {"workers": [st for _, _, st in zs]})\n    for st in ("oa", "fc"):\n        if all(st in p["R"] for p in parts):\n            ms = [p["R"][st][0] for p in parts]\n            out[st] = ({k: pd.concat([m[k] for m in ms if k in m]) for k in dict.fromkeys(k for m in ms for k in m)},\n                       {"failed": sum((p["R"][st][1].get("failed", []) for p in parts), []),\n                        "flipped": sum(p["R"][st][1].get("flipped", 0) for p in parts),\n                        "wall_s": max(p["R"][st][1].get("wall_s", 0) for p in parts)})\n    if all("syn" in p["R"] for p in parts):\n        out["syn"] = (pd.concat([p["R"]["syn"][0] for p in parts]),\n                      {"failed": sum((p["R"]["syn"][1].get("failed", []) for p in parts), []),\n                       "flipped": sum(p["R"]["syn"][1].get("flipped", 0) or 0 for p in parts)})\n    return out\n\n\ndef predict(root, studies, series, pkg, threads=None, workers=2, prof=False, work="/kaggle/working/fast_jobs", log=print,\n            orders=(("basin", "oa", "fc", "syn"), ("basin", "oa", "fc", "syn")), merge_crops=False, pad_loc=32, prof_sync=False):\n    """= ensemble.predict(root, studies, series, pkg, threads) with the four single-GPU stages on two GPUs. Returns its 4-tuple; FAST\n    holds the worker / stage timings. orders: the stage order per worker (independent stages: outputs do not depend on it)."""\n    bp, ki, oa_stage, fc_stage, syn_stage, win_stage = _mods(pkg)\n    import ensemble\n    import torch\n    if prof:\n        install_prof(pkg, "parent")\n    serial = os.environ.get("FAST_SERIAL") == "1"           # local parity: both workers on GPU 0, one after another\n    ngpu = torch.cuda.device_count()\n    assert serial or ngpu >= 2, f"fast.predict needs 2 GPUs, found {ngpu}"\n    os.makedirs(work, exist_ok=True)\n    t0 = time.time()\n    procs = []\n    for g, part in enumerate(split(studies, series)):\n        job, out = f"{work}/job{g}.pkl", f"{work}/out{g}.pkl"\n        pickle.dump({"studies": part, "series": series, "workers": workers, "prof": prof, "name": f"gpu{g}", "order": list(orders[g]), "merge_crops": merge_crops, "pad_loc": pad_loc, "prof_sync": prof_sync},\n                    open(job, "wb"))\n        if os.path.exists(out):\n            os.remove(out)\n        fh = open(f"{work}/worker{g}.log", "w")\n        env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0" if serial else str(g)}\n        procs.append((subprocess.Popen([sys.executable, __file__, "worker", pkg, root, job, out], env=env, stdout=fh, stderr=subprocess.STDOUT), out, fh, len(part)))\n        if serial:\n            procs[-1][0].wait()\n    for p, _, fh, _ in procs:\n        p.wait()\n        fh.close()\n    t_par = time.time() - t0\n    for g, (p, out, _, n) in enumerate(procs):\n        if p.returncode != 0 or not os.path.exists(out):\n            raise RuntimeError(f"fast worker gpu{g} failed (code {p.returncode}); log tail:\\n" + open(f"{work}/worker{g}.log").read()[-4000:])\n    parts = [pickle.load(open(out, "rb")) for _, out, _, _ in procs]\n    M = _merge(parts)\n    log(f"fast: workers done in {t_par:.0f}s ({\', \'.join(f\'gpu{g} {n} studies \' + str({k: round(v) for k, v in pp[\'T\'].items()}) for g, ((_, _, _, n), pp) in enumerate(zip(procs, parts)))})")\n    STAGE_LOG.extend(e for pp in parts for e in pp["stage_log"])\n\n    orig = {"bp": bp.predict, "oa": oa_stage.predict, "fc": fc_stage.predict_pkg, "syn": syn_stage.predict}\n\n    def bp_fake(*a, **k):\n        z, w, st = M["basin"]\n        bp.WIN.clear()\n        bp.WIN.update(w)\n        bp.STATS.clear()\n        bp.STATS.update(st)\n        return z\n\n    def oa_fake(*a, **k):\n        m, st = M["oa"]\n        oa_stage.STATS.clear()\n        oa_stage.STATS.update(st)\n        return dict(m)\n\n    def fc_fake(*a, **k):\n        m, st = M["fc"]\n        fc_stage.STATS.clear()\n        fc_stage.STATS.update(st)\n        return dict(m)\n\n    def syn_fake(*a, **k):\n        zm, st = M["syn"]\n        syn_stage.STATS.clear()\n        syn_stage.STATS.update(st)\n        return zm.copy()\n\n    bp.predict, oa_stage.predict, fc_stage.predict_pkg, syn_stage.predict = bp_fake, oa_fake, fc_fake, syn_fake\n    try:\n        with parent_release(ki):\n            res = ensemble.predict(root, studies, series, pkg, threads)\n    finally:\n        bp.predict, oa_stage.predict, fc_stage.predict_pkg, syn_stage.predict = orig["bp"], orig["oa"], orig["fc"], orig["syn"]\n    release()\n    FAST.clear()\n    FAST.update(t0=t0, parallel_s=t_par, parent_rss_gb=rss_gb(), workers=[{"n": n, **pp["T"]} for (_, _, _, n), pp in zip(procs, parts)], total_s=time.time() - t0,\n                main_s=ensemble.STATS.get("main", {}).get("wall_s"), win_s=ensemble.STATS.get("win", {}).get("wall_s"))\n    return res\n\n\nFAST = {}\n\nif __name__ == "__main__" and len(sys.argv) > 1 and sys.argv[1] == "worker":\n    worker(*sys.argv[2:6])\n')
open("/kaggle/working/dicom_prefetch.py", "w").write('"""DICOM read-ahead for the FAST workers (Submission Timing 10-22 reference, + Experts: hit-rate / resident-ahead stats; OAI: no).\nEXACT: it only reads files into the page cache; every consumer still reads and parses its files itself, so outputs are bit-identical.\n\nWhy:\n- Kaggle scoring VMs differ hugely in input-read latency. gold-58, same package: Basin wait_data 4-5 vs 160-212 s per worker per chunk\n  (~26 s / study on slow VMs).\n- Basin reads each study\'s files ONE BY ONE (DicomStudies.__getitem__ -> infer_prep._read = pydicom.dcmread per file, then\n  POSIX_FADV_DONTNEED) in only 2 DataLoader workers: latency-bound.\n- The crop stage and knee_infer\'s main pass already read in parallel (knee_infer.io_pool, 128 threads); the crop stage was unaffected.\nDesign:\n- A separate PREFETCHER PROCESS (no threads inside the FAST worker, which forks DataLoader workers: fork + threads can deadlock).\n  It reads the consumer\'s upcoming studies, file by file, with `threads` parallel readers (16 MB blocks), at most `ahead` studies\n  beyond the consumer\'s position.\n- Position: an 8-byte /dev/shm file. The consumer\'s Dataset.__getitem__(i) writes i (any process, fork-safe: pwrite). The prefetcher\n  takes the running max, so two loader workers writing i+1 then i is fine.\n- Basin\'s post-read eviction does not fight the prefetch: it evicts files Basin has already consumed; the prefetch window is ahead of\n  them. A bounded window keeps the page-cache use at ~ahead x study size (~24 x 120 MB = ~3 GB).\n- Failure mode: if the prefetcher dies, consumers read as before (slower, same outputs); stop() reports its exit code / stats.\nUse (per stage, in the FAST worker):\n    pf = start(root, studies_in_consumer_order, threads=64, ahead=24, tag="basin")   # the prefetcher lists studies itself\n    unhook = hook_getitem(bp.DicomStudies, pf)          # must run BEFORE the stage creates its DataLoader (workers fork after)\n    try: ...stage...  finally: unhook(); stats = stop(pf)\nPrefetcher process: python dicom_prefetch.py PROGRESS_PATH JOBS_JSON THREADS AHEAD STATS_JSON"""\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\n\nBLOCK = 16 << 20\n\n\ndef study_groups(root, studies):\n    """Every file of each study (sorted), in the consumer\'s study order."""\n    return [sorted(os.path.join(d, f) for d, _, fs in os.walk(os.path.join(root, s)) for f in fs) for s in studies]\n\n\ndef set_progress(path, i):\n    fd = os.open(path, os.O_WRONLY)\n    try:\n        os.pwrite(fd, int(i).to_bytes(8, "little", signed=True), 0)\n    finally:\n        os.close(fd)\n\n\ndef get_progress(path):\n    with open(path, "rb", buffering=0) as fh:\n        return int.from_bytes(fh.read(8), "little", signed=True)\n\n\ndef start(root, studies, threads=64, ahead=24, tag="pf", log=None):\n    """Prefetch root/<sid>/** for `studies` (the consumer\'s order). The PREFETCHER lists each study lazily (lister threads) when it\n    enters the window, so the consumer process does no up-front directory walk (slow on a high-latency mount)."""\n    base = f"/dev/shm/knee_pf_{os.getpid()}_{tag}"\n    with open(base + ".pos", "wb") as fh:\n        fh.write((-1).to_bytes(8, "little", signed=True))\n    json.dump({"root": root, "studies": list(studies)}, open(base + ".jobs.json", "w"))\n    groups = studies\n    out = open(log or base + ".log", "w")\n    p = subprocess.Popen([sys.executable, os.path.abspath(__file__), base + ".pos", base + ".jobs.json", str(threads), str(ahead),\n                          base + ".stats.json"], stdout=out, stderr=subprocess.STDOUT)\n    return {"proc": p, "base": base, "out": out, "t0": time.time(), "n": len(groups)}\n\n\ndef hook_getitem(cls, pf):\n    """Wrap cls.__getitem__ so every item request first publishes its index. Returns the unhook function."""\n    orig, path = cls.__getitem__, pf["base"] + ".pos"\n\n    def gi(self, i):\n        set_progress(path, i)\n        return orig(self, i)\n    gi.__wrapped__ = orig\n    cls.__getitem__ = gi\n\n    def unhook():\n        cls.__getitem__ = orig\n    return unhook\n\n\ndef stop(pf, wait=5.0):\n    """Ask the prefetcher to finish (it exits when the consumer passed the last study) or kill it after `wait` s. -> stats dict."""\n    set_progress(pf["base"] + ".pos", 1 << 40)\n    try:\n        pf["proc"].wait(wait)\n    except subprocess.TimeoutExpired:\n        pf["proc"].kill()\n        pf["proc"].wait()\n    pf["out"].close()\n    st = {"returncode": pf["proc"].returncode, "wall_s": time.time() - pf["t0"], "studies": pf["n"]}\n    try:\n        st |= json.load(open(pf["base"] + ".stats.json"))\n    except (OSError, ValueError):\n        pass\n    for ext in (".pos", ".jobs.json", ".stats.json", ".log"):\n        try:\n            os.remove(pf["base"] + ext)\n        except OSError:\n            pass\n    return st\n\n\ndef _read(path):\n    n = 0\n    try:\n        with open(path, "rb", buffering=0) as fh:\n            while True:\n                b = fh.read(BLOCK)\n                if not b:\n                    return n\n                n += len(b)\n    except OSError:\n        return n\n\n\nREWIND = 16   # a published index this far below the running max = a NEW pass over the same studies (Basin: one pass per model group);\n#             2 loader workers x prefetch 2 batches x bs 2 reorder by < 16\n\n\ndef _list(root, sid):\n    return sorted(os.path.join(d, f) for d, _, fs in os.walk(os.path.join(root, sid)) for f in fs)\n\n\ndef _main(pos, jobs, threads, ahead, stats):\n    J = json.load(open(jobs))\n    root, groups = J["root"], J["studies"]\n    t0, seen, nxt, nbytes, nfiles, max_ahead, passes = time.time(), -1, 0, 0, 0, 0, 1\n    pending, listing = [], []\n    hits = n_pos = res_sum = 0       # Experts (Master 08:3x): per new consumer position i, HIT = study i fully read when the position was\n    futs = {}                        # seen (polled every 50 ms: a mild upper bound); resident_ahead = fully read studies right after i\n    resident = lambda k: k in futs and all(f.done() for f in futs[k])    # noqa: E731\n    with ThreadPoolExecutor(threads) as pool, ThreadPoolExecutor(8) as lister:\n        while True:                                                  # until stop(): consumers may start further passes\n            cur = get_progress(pos)\n            if cur >= 1 << 40:\n                break\n            if 0 <= cur < seen - REWIND:\n                seen, nxt, passes = cur, cur + 1, passes + 1\n                futs = {}                                            # a new pass: residency is tracked afresh\n            elif cur > seen:                                         # the consumer reached new studies (this pass)\n                for k in range(max(seen + 1, 0), cur + 1):\n                    hits += resident(k)\n                    n_pos += 1\n                r = 0\n                while resident(cur + 1 + r):\n                    r += 1\n                res_sum += r\n            seen = max(seen, cur)\n            lo = max(seen, 0)\n            nxt = max(nxt, lo + 1 if seen >= 0 else 0)           # never read what the consumer already has\n            while nxt < len(groups) and nxt < lo + 1 + ahead:\n                listing.append((nxt, lister.submit(_list, root, groups[nxt])))   # in order; reads start as soon as each listing lands\n                nxt += 1\n            max_ahead = max(max_ahead, nxt - lo)\n            while listing and listing[0][1].done():\n                k, li = listing.pop(0)\n                fs = li.result()\n                futs[k] = [pool.submit(_read, f) for f in fs]\n                pending += futs[k]\n                nfiles += len(fs)\n            done = [f for f in pending if f.done()]\n            nbytes += sum(f.result() for f in done)\n            pending = [f for f in pending if not f.done()]\n            time.sleep(0.05)\n        for _, li in listing:\n            li.cancel()\n        nbytes += sum(f.result() for f in pending if not f.cancelled())\n    json.dump({"prefetched_studies": nxt, "files": nfiles, "gb": nbytes / 2**30, "s": time.time() - t0, "max_ahead": max_ahead,\n               "passes": passes, "positions": n_pos, "hit_rate": hits / max(n_pos, 1), "resident_ahead_mean": res_sum / max(n_pos, 1)},\n              open(stats, "w"))\n\n\nif __name__ == "__main__":\n    _main(sys.argv[1], sys.argv[2], int(sys.argv[3]), int(sys.argv[4]), sys.argv[5])\n')
sys.path.insert(0, "/kaggle/working")
import fast
FAST_CHUNKS, FAST_PAD_LOC = 4, 32


def strict_check(ids, failed, parts, tag):
    """Raise if any study-level fallback of the package fired in the last predict: the stage failure counters (ensemble.STATS, incl.
    Basin's per-worker stats in FAST mode = fast._merge's {"workers": [...]}; Brainstorm review 10-20), the failed main studies, and
    any part with a NaN / missing study (every inherited fillna path: a stage returning NaN for a study it did not list as failed)."""
    S, bad = ensemble.STATS, []
    if len(failed):
        bad.append(f"main failed {len(failed)}: {list(failed)[:5]}")
    for k, f in (("main_sd", "failed"), ("win", "missing"), ("oa", "failed"), ("fc", "failed"), ("syn", "failed")):
        n = (S.get(k) or {}).get(f, 0)
        if n:
            bad.append(f"{k} {f} {n}")
    b = S.get("basin") or {}
    sts = b["workers"] if "workers" in b else [b]          # FAST: one bp.STATS per worker
    series_level = []
    for i, st in enumerate(sts):
        assert "nonfinite" in st and any(g.startswith("group") for g in st), f"STRICT: basin STATS {i} without counters: {list(st)}"
        if st["nonfinite"] or st["det_nonfinite"]:
            bad.append(f"basin worker {i} non-finite {st['nonfinite']} / detector {st['det_nonfinite']}")
        for g, v in st.items():
            if g.startswith("group"):
                nf = [int(x) for x in v["nfail"]]
                if nf[0]:
                    bad.append(f"basin worker {i} {g}: study scans failed {nf[0]}")
                series_level.append(nf[1:])
    for k, v in parts.items():
        v = v.to_frame() if isinstance(v, pd.Series) else v
        n = int(v.reindex(ids).isna().any(axis=1).sum())
        if n:
            bad.append(f"part {k}: {n} studies NaN / missing")
    if bad:
        raise RuntimeError(f"STRICT ({tag}): a study-level fallback fired -> stop loudly: " + "; ".join(bad))
    print(f"STRICT ({tag}): no fallback fired ({len(parts)} parts checked; Basin series-level [skipped at scan, failed, frames replaced] "
          f"per group {series_level})", flush=True)


def fast_predict_all(root, ids, series, n_chunks=FAST_CHUNKS):
    ids = list(ids)
    miss = sorted(set(ids) - set(series.StudyInstanceUID))
    if miss:
        raise RuntimeError(f"STRICT: {len(miss)} studies have no rows in the series csv: {miss[:5]}")
    if len(ids) <= 3:   # Kaggle's commit run (3 sample studies): the package's own sequential predict, as every scored kernel did
        ensemble.STATS.clear()
        r = ensemble.predict(root, ids, series, PKG)
        strict_check(ids, r[2], r[1], "sequential <= 3 studies")
        return r + ([{"mode": "sequential", "n": len(ids)}],)
    ngpu = torch.cuda.device_count()
    if ngpu < 2:
        raise RuntimeError(f"STRICT: the FAST runner needs 2 GPUs, found {ngpu} (no silent switch to the sequential runner)")
    chunks = [ids]
    while len(chunks) < n_chunks:
        chunks = [c for ch in chunks for c in fast.split(ch, series)]
    assert sorted(s for c in chunks for s in c) == sorted(ids), "chunks must cover the ids exactly once"
    print(f"FAST STRICT: {len(ids)} studies in {len(chunks)} chunk(s) {[len(c) for c in chunks]}, pad_loc {FAST_PAD_LOC}", flush=True)
    R, LOG = [], []
    for i, ch in enumerate(chunks):
        t = time.time()
        ensemble.STATS.clear()
        r = fast.predict(root, ch, series, PKG, workers=2, work=f"/kaggle/working/fast_jobs{i}",
                         orders=(("basin", "oa", "fc", "syn"), ("oa", "fc", "syn", "basin")), merge_crops=True, pad_loc=FAST_PAD_LOC)
        strict_check(ch, r[2], r[1], f"chunk {i}")
        R.append(r)
        LOG.append({"chunk": i, "n": len(ch), "mode": "fast", "wall_s": time.time() - t, "fast": json.loads(json.dumps(fast.FAST, default=str))})
        print(f"FAST chunk {i}: {len(ch)} studies in {time.time() - t:.0f}s ({time.time() - T0:.0f}s since start), parent rss "
              f"{fast.FAST.get('parent_rss_gb', 0):.1f} GB", flush=True)
        gc.collect()
        fast.release()
    pred = pd.concat([r[0] for r in R])
    parts = {k: pd.concat([r[1][k] for r in R if k in r[1]]) for k in dict.fromkeys(k for r in R for k in r[1])}
    assert pred.index.is_unique and set(pred.index) == set(ids), "FAST: predictions must cover the ids exactly once"
    return pred, parts, [s for r in R for s in r[2]], pd.concat([pd.Series(r[3]) for r in R]), LOG


if len(test) <= 3:   # commit run only: the FINAL form on 24 train studies of the check list, BEFORE anything else heavy in this process
    chk = pd.read_csv(PKG + "/check_studies.csv")
    CHK_IDS = chk.StudyInstanceUID[chk.group == "edge"].tolist()[:4]   # QUICK commit (make_kernel --quick 4)   # v2: the 8 edge studies (v1's run covered 24 incl. these for parity)
    t = time.time()
    cp, cps, cf, cs, CLOG = fast_predict_all(COMP + "/train_series", CHK_IDS, pd.read_csv(COMP + "/train_series.csv"), n_chunks=1)
    cp.to_csv("check_pred_base.csv")
    decouple(cp).to_csv("check_pred.csv")
    pd.concat(cps, names=["part"]).to_csv("check_parts.csv")
    json.dump({"n": len(CHK_IDS), "wall_s": time.time() - t, "fast_log": CLOG, "stats_last_chunk": ensemble.STATS, "total_s": time.time() - T0},
              open("check_timing.json", "w"), indent=1, default=str)
    print(f"FINAL CHECK: {len(CHK_IDS)} studies in {time.time() - t:.0f}s, strict OK", flush=True)
    del cp, cps
    gc.collect()
    fast.release()
pred, parts, failed, secs, FAST_LOG = fast_predict_all(COMP + "/test_series", test.StudyInstanceUID.tolist(), pd.read_csv(COMP + "/test_series.csv"))
print(f"test: {len(pred)} studies in {time.time() - t:.0f}s, failed main {len(failed)}, {ensemble.STATS}", flush=True)
sub = pd.read_csv(COMP + "/sample_submission.csv")
assert set(sub.StudyInstanceUID) == set(test.StudyInstanceUID) and len(sub) == len(test)
sub[ki.FINDINGS] = pred.loc[sub.StudyInstanceUID, ki.FINDINGS].to_numpy()
sub[ki.FINDINGS] = decouple(sub[ki.FINDINGS])
print("de-coupled:", list(DEC), flush=True)
assert np.isfinite(sub[ki.FINDINGS].to_numpy()).all() and sub[ki.FINDINGS].to_numpy().min() >= 0 and sub[ki.FINDINGS].to_numpy().max() <= 1
KEEP = []                                              # probe: every other finding is a constant 0.5 (AUC exactly 0.5)
if KEEP:
    sub[[c for c in ki.FINDINGS if c not in KEEP]] = 0.5
    assert (sub[[c for c in ki.FINDINGS if c not in KEEP]] == 0.5).all().all() and sub[KEEP].nunique().min() > 1
sub.to_csv("submission.csv", index=False)
print(f"submission.csv written, {time.time() - T0:.0f}s since start", flush=True)

if False:  # FINAL kernel: Experts' sequential check block replaced by the FINAL-form check above
    chk = pd.read_csv(PKG + "/check_studies.csv")
    series = pd.read_csv(COMP + "/train_series.csv")
    timing, preds, all_parts = {"ncpu": NCPU, "gpus": torch.cuda.device_count()}, [], []
    for name, ids in [("main", chk.StudyInstanceUID[chk.group != "edge"].tolist()), ("edge", chk.StudyInstanceUID[chk.group == "edge"].tolist())]:
        t = time.time()
        p, ps, f, s = ensemble.predict(COMP + "/train_series", ids, series, PKG)
        timing[name] = {"n": len(ids), "wall_s": time.time() - t, "cpu_s_per_study": float(s.mean()), "failed": f,
                        **{k: ensemble.STATS.get(k) for k in ensemble.STATS}}      # every stage incl. main_sd
        preds.append(p)
        all_parts.append(pd.concat(ps, names=["part"]))
        print(name, timing[name], flush=True)
    pd.concat(preds).to_csv("check_pred_base.csv")
    decouple(pd.concat(preds)).to_csv("check_pred.csv")
    pd.concat(all_parts).to_csv("check_parts.csv")
    timing["files_per_study_main"] = float(np.mean([sum(1 for e in os.scandir(f"{COMP}/train_series/{st}") for _ in os.scandir(e.path))
                                                    for st in chk.StudyInstanceUID[chk.group != "edge"]]))
    timing["total_s"] = time.time() - T0
    json.dump(timing, open("check_timing.json", "w"), indent=1)
    print(timing, flush=True)